# RaDe-GS vs. RaDe-GS + TSDF-prior — Colab experiment

Runs `examples/experiment_tsdf_prior.py` on a CUDA Colab runtime against a DTU scene from the 2DGS+COLMAP bundle.

**Important:** the local `mpsplat` repo is an Apple-Silicon-only fork with the CUDA backend stripped. On Colab we install **upstream `gsplat`** (from PyPI) and hide the local stripped copy so imports resolve to the CUDA build. Everything else — `methods/rade_gs.py`, `methods/tsdf_prior.py`, the experiment script — comes from the repo.

Edit the `SCENE` cell below if you want a different DTU scan.

## 1. Pick GPU and scene

In [ ]:
SCENE         = "scan24"
DATA_FACTOR   = 2          # half-res, matches RaDe-GS Table 1 main DTU rows
MAX_STEPS     = 30000
DRIVE_DATA    = "/content/drive/MyDrive/mpsplat_data/2DGS_data/dtu.tar.gz"
DRIVE_RESULTS = f"/content/drive/MyDrive/mpsplat_results/{SCENE}"

!nvidia-smi --query-gpu=name,memory.total,memory.used,utilization.gpu --format=csv

## 2. Clone (or update) the mpsplat repo

In [ ]:
import os
if not os.path.isdir("/content/mpsplat"):
    !cd /content && git clone https://github.com/zacsmms/mpsplat.git
else:
    !cd /content/mpsplat && git pull origin main

## 3. Install upstream `gsplat` (CUDA) + experiment deps

Hide the local stripped `gsplat/` directory so `from gsplat import ...` resolves to the CUDA-backed PyPI install. Skip the heavy CUDA-build extras in `examples/requirements.txt` — we only need the dataset loader's deps.

In [ ]:
import shutil, os
local_gsplat = "/content/mpsplat/gsplat"
shadow = "/content/mpsplat/.gsplat_local_stripped"
if os.path.isdir(local_gsplat) and not os.path.isdir(shadow):
    shutil.move(local_gsplat, shadow)
    print("renamed local stripped gsplat ->", shadow)
else:
    print("local gsplat already shadowed (or absent)")

!pip uninstall -y mpsplat pycolmap 2>/dev/null | tail -2
!pip install -q gsplat
!pip install -q \
    "git+https://github.com/rmbrualla/pycolmap@cc7ea4b7301720ac29287dbe450952511b32125e" \
    "numpy<2.0.0" scipy scikit-learn tqdm "torchmetrics[image]" \
    opencv-python tyro Pillow piexif matplotlib imageio

## 4. Verify the upstream `gsplat` is CUDA-backed and has `extra_signals`

In [ ]:
import sys, inspect, importlib
for k in list(sys.modules):
    if k == "gsplat" or k.startswith("gsplat."):
        del sys.modules[k]
import gsplat
print("gsplat loaded from:", gsplat.__file__)
sig = inspect.signature(gsplat.rasterization).parameters
print("has extra_signals:", "extra_signals" in sig)
assert "extra_signals" in sig, (
    "Upstream gsplat is too old; rade_gs.py needs `extra_signals=`. "
    "Try: pip install -q 'gsplat>=1.4.0'  (or install the latest from git)."
)

## 5. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 6. Extract the DTU bundle to local Colab disk

In [ ]:
%cd /content/mpsplat
!mkdir -p data
import os, pathlib
assert os.path.exists(DRIVE_DATA), f"can't find {DRIVE_DATA}"
# Skip if the scene is already on local disk from a prior session.
if not any(pathlib.Path("data").rglob(SCENE)):
    !tar -xzf "$DRIVE_DATA" -C data/
else:
    print("data already extracted")
!find data -maxdepth 3 -name "$SCENE" -type d

## 7. Resolve the scene path (case-sensitive on Linux)

The 2DGS bundle uses `DTU/` (uppercase). We let Python find the actual path so the rest of the notebook works regardless.

In [ ]:
import pathlib
scene_dirs = list(pathlib.Path("data").rglob(SCENE))
assert scene_dirs, f"scene {SCENE} not found under data/"
SCENE_DIR = str(scene_dirs[0])
print("SCENE_DIR =", SCENE_DIR)
!ls "$SCENE_DIR" && echo --- && ls "$SCENE_DIR/sparse/0"

## 8. Generate `images_2/` if missing

The COLMAP loader expects `images_{factor}/` to already exist. The 2DGS bundle only ships full-res `images/`.

In [ ]:
from PIL import Image
import os, glob
src = f"{SCENE_DIR}/images"
dst = f"{SCENE_DIR}/images_{DATA_FACTOR}"
if os.path.isdir(dst) and len(os.listdir(dst)) == len(os.listdir(src)):
    print("already exists:", dst, "(", len(os.listdir(dst)), "images )")
else:
    os.makedirs(dst, exist_ok=True)
    for p in sorted(glob.glob(f"{src}/*")):
        img = Image.open(p)
        w, h = img.width // DATA_FACTOR, img.height // DATA_FACTOR
        img.resize((w, h), Image.LANCZOS).save(f"{dst}/{os.path.basename(p)}")
    print("wrote", len(os.listdir(dst)), "images to", dst)
from PIL import Image as I
print("full size :", I.open(sorted(glob.glob(f'{src}/*'))[0]).size)
print("down size :", I.open(sorted(glob.glob(f'{dst}/*'))[0]).size)

## 9. Rasterizer timing sanity check

On A100 this should be tens of ms per forward; on T4 sub-second. If it's seconds-per-forward, CUDA isn't being used and the experiment will be unusably slow.

In [ ]:
import time, sys, torch
sys.path.insert(0, "/content/mpsplat")
from methods.rade_gs import rasterization_rade_gs

N, H, W = 100_000, 600, 800
device = "cuda"
torch.manual_seed(0)
means     = torch.randn(N, 3, device=device, requires_grad=True)
quats     = torch.zeros(N, 4, device=device); quats[:, 0] = 1
scales    = torch.full((N, 3), 0.01, device=device, requires_grad=True)
opacities = torch.full((N,), 0.5, device=device, requires_grad=True)
colors    = torch.rand(N, 3, device=device, requires_grad=True)
viewmats  = torch.eye(4, device=device).unsqueeze(0)
K         = torch.tensor([[800., 0, 400], [0, 800, 300], [0, 0, 1]], device=device).unsqueeze(0)

# Warm-up.
for _ in range(3): _ = rasterization_rade_gs(means, quats, scales, opacities, colors, viewmats, K, W, H)
torch.cuda.synchronize(); t0 = time.time()
for _ in range(10): _ = rasterization_rade_gs(means, quats, scales, opacities, colors, viewmats, K, W, H)
torch.cuda.synchronize()
print(f"{(time.time() - t0) / 10 * 1000:.1f} ms / forward  (N={N}, {H}x{W})")

## 10. Run the experiment

In [ ]:
import os
os.makedirs(DRIVE_RESULTS, exist_ok=True)

!python examples/experiment_tsdf_prior.py \
    --data_dir "$SCENE_DIR" \
    --result_dir "$DRIVE_RESULTS" \
    --data_factor $DATA_FACTOR \
    --max_steps $MAX_STEPS \
    2>&1 | tee "$DRIVE_RESULTS/train.log"

## 11. Inspect comparison report

In [ ]:
from IPython.display import Markdown, Image, display
import os
cmp = f"{DRIVE_RESULTS}/comparison.md"
if os.path.exists(cmp):
    display(Markdown(open(cmp).read()))
else:
    print("no comparison.md yet — check the train.log cell for errors")

In [ ]:
from IPython.display import Image, display, Markdown
import os
for kind in ["rgb", "depth", "normal"]:
    for method in ["baseline", "tsdf_prior"]:
        p = f"{DRIVE_RESULTS}/{method}/renders/000_{kind}.png"
        if os.path.exists(p):
            display(Markdown(f"**{method} / {kind}**"))
            display(Image(filename=p))
        else:
            print("missing:", p)

## 12. (Optional) sweep `tsdf_lambda`

Once baseline + default-tsdf are reproducible, scan the prior weight without retraining baseline each time.

In [ ]:
for lam in [0.025, 0.05, 0.10, 0.20]:
    out = f"{DRIVE_RESULTS}_lam{lam:g}"
    print("=>", out)
    !python examples/experiment_tsdf_prior.py \
        --data_dir "$SCENE_DIR" \
        --result_dir "$out" \
        --data_factor $DATA_FACTOR \
        --max_steps $MAX_STEPS \
        --methods tsdf_prior \
        --tsdf_lambda $lam